# Notebook 06: Full End-to-End Pipeline

**Goal:** Wire together the best option from each stage into a single end-to-end pipeline that takes any input (PDF/image) and produces translated output.

**Pipeline:** Input → Page Images → OCR → Translate → Inpaint → Overlay → Output

This notebook uses the best choices identified in notebooks 02-05. Modify the configuration to use different engines.

In [ ]:
# Install all dependencies (run once)
# !pip install pdf2image Pillow opencv-python-headless numpy matplotlib pytesseract easyocr anthropic tqdm
# !sudo apt-get install -y poppler-utils tesseract-ocr
# !python scripts/download_fonts.py

In [ ]:
import io
import json
import os
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from tqdm.notebook import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    DATA_DIR, OUTPUT_DIR, FONTS_DIR, SAMPLES_DIR,
    save_json, load_json, save_image, load_image,
    display_images, display_comparison,
    pil_to_cv2, cv2_to_pil,
    sample_background_color, estimate_text_color,
)

print("All imports loaded successfully.")

## Configuration

Set your input file and target languages. Choose the best engine for each stage based on your experiments in notebooks 02-05.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit these settings
# ══════════════════════════════════════════════════════════════════════════════

# Input file (PDF or image)
INPUT_FILE = SAMPLES_DIR / "sample.pdf"

# Target languages to translate into
TARGET_LANGUAGES = ["hindi"]  # Add more: ["hindi", "tamil", "bengali", "punjabi", "gujarati"]

# Page range (None = all pages)
PAGE_RANGE = None  # e.g., [0, 1, 2] for first 3 pages

# DPI for PDF rendering
DPI = 300

# Stage choices (from your experiments in notebooks 02-05)
OCR_ENGINE = "easyocr"           # Options: "tesseract", "easyocr", "google_vision", "azure_vision"
TRANSLATOR_ENGINE = "claude"     # Options: "claude", "google_translate", "nllb", "indictrans2"
INPAINTING_METHOD = "opencv_telea"  # Options: "opencv_telea", "opencv_ns", "lama", "simple_fill"

# Output directory
PIPELINE_OUTPUT_DIR = OUTPUT_DIR / "pipeline"

print("Configuration:")
print(f"  Input: {INPUT_FILE}")
print(f"  Languages: {TARGET_LANGUAGES}")
print(f"  OCR: {OCR_ENGINE}")
print(f"  Translator: {TRANSLATOR_ENGINE}")
print(f"  Inpainting: {INPAINTING_METHOD}")

## Pipeline Components

Self-contained functions for each stage, copied from the individual notebooks with the best choices.

In [ ]:
# ── Language config ────────────────────────────────────────────────────────────

LANGUAGE_CODES = {
    "hindi":    {"google": "hi", "nllb": "hin_Deva", "indictrans": "hin_Deva", "name": "Hindi"},
    "tamil":    {"google": "ta", "nllb": "tam_Taml", "indictrans": "tam_Taml", "name": "Tamil"},
    "bengali":  {"google": "bn", "nllb": "ben_Beng", "indictrans": "ben_Beng", "name": "Bengali"},
    "punjabi":  {"google": "pa", "nllb": "pan_Guru", "indictrans": "pan_Guru", "name": "Punjabi"},
    "gujarati": {"google": "gu", "nllb": "guj_Gujr", "indictrans": "guj_Gujr", "name": "Gujarati"},
}

LANGUAGE_FONT_MAP = {
    "hindi":    "NotoSansDevanagari-Regular.ttf",
    "tamil":    "NotoSansTamil-Regular.ttf",
    "bengali":  "NotoSansBengali-Regular.ttf",
    "punjabi":  "NotoSansGurmukhi-Regular.ttf",
    "gujarati": "NotoSansGujarati-Regular.ttf",
}

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp", ".webp"}


# ── Stage 0: Input Loading ────────────────────────────────────────────────────

def load_input(file_path: Path, dpi: int = 300) -> list[Image.Image]:
    """Load PDF or image file as list of PIL Images."""
    file_path = Path(file_path)
    ext = file_path.suffix.lower()
    
    if ext == ".pdf":
        from pdf2image import convert_from_path
        return convert_from_path(str(file_path), dpi=dpi)
    elif ext in IMAGE_EXTENSIONS:
        return [Image.open(str(file_path)).convert("RGB")]
    else:
        raise ValueError(f"Unsupported file type: {ext}")


# ── Stage 1: OCR ──────────────────────────────────────────────────────────────

def run_ocr(image: Image.Image, engine: str = "easyocr") -> list[dict]:
    """Run OCR and return standardized text blocks."""
    
    if engine == "tesseract":
        import pytesseract
        from pytesseract import Output
        data = pytesseract.image_to_data(image, lang="eng", output_type=Output.DICT)
        lines = {}
        for i in range(len(data["text"])):
            text = data["text"][i].strip()
            conf = int(data["conf"][i])
            if conf < 0 or not text:
                continue
            key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
            if key not in lines:
                lines[key] = {"words": [], "x0": data["left"][i], "y0": data["top"][i],
                              "x1": data["left"][i] + data["width"][i], "y1": data["top"][i] + data["height"][i], "confs": []}
            line = lines[key]
            line["words"].append(text)
            line["confs"].append(conf)
            line["x0"] = min(line["x0"], data["left"][i])
            line["y0"] = min(line["y0"], data["top"][i])
            line["x1"] = max(line["x1"], data["left"][i] + data["width"][i])
            line["y1"] = max(line["y1"], data["top"][i] + data["height"][i])
        blocks = []
        for idx, (_, line) in enumerate(sorted(lines.items())):
            blocks.append({"id": idx, "text": " ".join(line["words"]),
                          "bbox": [line["x0"], line["y0"], line["x1"], line["y1"]],
                          "confidence": round(np.mean(line["confs"]) / 100.0, 3), "level": "line"})
        return blocks
    
    elif engine == "easyocr":
        import easyocr
        reader = easyocr.Reader(["en"], gpu=False)
        results = reader.readtext(np.array(image))
        blocks = []
        for idx, (polygon, text, confidence) in enumerate(results):
            xs = [p[0] for p in polygon]
            ys = [p[1] for p in polygon]
            blocks.append({"id": idx, "text": text,
                          "bbox": [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))],
                          "confidence": round(float(confidence), 3), "level": "line"})
        return blocks
    
    elif engine == "google_vision":
        from google.cloud import vision
        client = vision.ImageAnnotatorClient()
        buf = io.BytesIO(); image.save(buf, format="PNG")
        response = client.document_text_detection(image=vision.Image(content=buf.getvalue()))
        blocks = []
        idx = 0
        for page in response.full_text_annotation.pages:
            for block in page.blocks:
                for para in block.paragraphs:
                    words = ["".join(s.text for s in w.symbols) for w in para.words]
                    v = para.bounding_box.vertices
                    blocks.append({"id": idx, "text": " ".join(words),
                                  "bbox": [min(p.x for p in v), min(p.y for p in v), max(p.x for p in v), max(p.y for p in v)],
                                  "confidence": round(float(getattr(para, 'confidence', 0.9)), 3), "level": "paragraph"})
                    idx += 1
        return blocks
    
    elif engine == "azure_vision":
        from azure.ai.vision.imageanalysis import ImageAnalysisClient
        from azure.ai.vision.imageanalysis.models import VisualFeatures
        from azure.core.credentials import AzureKeyCredential
        client = ImageAnalysisClient(endpoint=os.environ["AZURE_VISION_ENDPOINT"],
                                      credential=AzureKeyCredential(os.environ["AZURE_VISION_KEY"]))
        buf = io.BytesIO(); image.save(buf, format="PNG")
        result = client.analyze(image_data=buf.getvalue(), visual_features=[VisualFeatures.READ])
        blocks = []
        idx = 0
        if result.read and result.read.blocks:
            for block in result.read.blocks:
                for line in block.lines:
                    pts = line.bounding_polygon
                    blocks.append({"id": idx, "text": line.text,
                                  "bbox": [int(min(p.x for p in pts)), int(min(p.y for p in pts)),
                                           int(max(p.x for p in pts)), int(max(p.y for p in pts))],
                                  "confidence": round(float(np.mean([w.confidence for w in line.words])), 3), "level": "line"})
                    idx += 1
        return blocks
    
    else:
        raise ValueError(f"Unknown OCR engine: {engine}")


print("OCR stage ready.")

In [ ]:
# ── Stage 2: Translation ──────────────────────────────────────────────────────

def translate_blocks(text_blocks: list[dict], target_lang: str, engine: str = "claude") -> list[dict]:
    """Translate text blocks using the chosen engine."""
    import anthropic
    
    lang_name = LANGUAGE_CODES[target_lang]["name"]
    
    if engine == "claude":
        client = anthropic.Anthropic()
        input_blocks = [{"id": b["id"], "text": b["text"]} for b in text_blocks if b["text"].strip()]
        
        system_prompt = f"""You are a professional translator specializing in educational textbook content.
Translate the following text segments from English to {lang_name}.
Rules:
- Maintain the educational tone and register
- For dialogue, keep it natural and conversational in {lang_name}
- Preserve numbers and proper nouns as appropriate
- Return ONLY a valid JSON array: [{{"id": N, "translated": "..."}}]"""
        
        response = client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=4096,
            system=system_prompt,
            messages=[{"role": "user", "content": json.dumps(input_blocks, ensure_ascii=False)}],
        )
        response_text = response.content[0].text.strip()
        if response_text.startswith("```"):
            response_text = response_text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        translated = json.loads(response_text)
        
        id_to_block = {b["id"]: b for b in text_blocks}
        return [{"id": t["id"], "original": id_to_block[t["id"]]["text"],
                 "translated": t["translated"], "bbox": id_to_block[t["id"]]["bbox"]}
                for t in translated if t["id"] in id_to_block]
    
    elif engine == "google_translate":
        from google.cloud import translate_v2 as translate
        client = translate.Client()
        code = LANGUAGE_CODES[target_lang]["google"]
        return [{"id": b["id"], "original": b["text"],
                 "translated": client.translate(b["text"], target_language=code, source_language="en")["translatedText"],
                 "bbox": b["bbox"]} for b in text_blocks if b["text"].strip()]
    
    elif engine == "nllb":
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        code = LANGUAGE_CODES[target_lang]["nllb"]
        tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
        model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
        tokenizer.src_lang = "eng_Latn"
        results = []
        for b in text_blocks:
            if not b["text"].strip(): continue
            inputs = tokenizer(b["text"], return_tensors="pt", max_length=512, truncation=True)
            tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(code), max_length=512)
            results.append({"id": b["id"], "original": b["text"],
                           "translated": tokenizer.decode(tokens[0], skip_special_tokens=True), "bbox": b["bbox"]})
        return results
    
    elif engine == "indictrans2":
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        code = LANGUAGE_CODES[target_lang]["indictrans"]
        tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-en-indic-dist-200M", trust_remote_code=True)
        model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-en-indic-dist-200M", trust_remote_code=True)
        tokenizer.src_lang = "eng_Latn"
        results = []
        for b in text_blocks:
            if not b["text"].strip(): continue
            inputs = tokenizer(b["text"], return_tensors="pt", max_length=512, truncation=True)
            tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(code), max_length=512)
            results.append({"id": b["id"], "original": b["text"],
                           "translated": tokenizer.decode(tokens[0], skip_special_tokens=True), "bbox": b["bbox"]})
        return results
    
    else:
        raise ValueError(f"Unknown translator: {engine}")


print("Translation stage ready.")

In [ ]:
# ── Stage 3: Inpainting ───────────────────────────────────────────────────────

def create_text_mask(image_size: tuple[int, int], text_blocks: list[dict],
                     dilation_px: int = 3, padding: int = 2) -> np.ndarray:
    """Create binary mask for text regions."""
    w, h = image_size
    mask = np.zeros((h, w), dtype=np.uint8)
    for block in text_blocks:
        x0, y0, x1, y1 = block["bbox"]
        x0, y0 = max(0, int(x0) - padding), max(0, int(y0) - padding)
        x1, y1 = min(w, int(x1) + padding), min(h, int(y1) + padding)
        mask[y0:y1, x0:x1] = 255
    if dilation_px > 0:
        kernel = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    return mask


def inpaint_image(image: Image.Image, text_blocks: list[dict], method: str = "opencv_telea") -> Image.Image:
    """Remove text from image using chosen inpainting method."""
    mask = create_text_mask(image.size, text_blocks)
    
    if method in ("opencv_telea", "opencv_ns"):
        img_cv = pil_to_cv2(image)
        flag = cv2.INPAINT_TELEA if method == "opencv_telea" else cv2.INPAINT_NS
        result = cv2.inpaint(img_cv, mask, inpaintRadius=7, flags=flag)
        return cv2_to_pil(result)
    
    elif method == "lama":
        from simple_lama_inpainting import SimpleLama
        lama = SimpleLama()
        mask_pil = Image.fromarray(mask).convert("L")
        return lama(image, mask_pil)
    
    elif method == "simple_fill":
        result = image.copy()
        draw = ImageDraw.Draw(result)
        for block in text_blocks:
            bbox = [int(c) for c in block["bbox"]]
            bg_color = sample_background_color(image, bbox)
            draw.rectangle(bbox, fill=bg_color)
        return result
    
    else:
        raise ValueError(f"Unknown inpainting method: {method}")


# ── Stage 4: Text Overlay (with clipping to prevent overflow) ─────────────

def get_font(language: str, size: int = 24) -> ImageFont.FreeTypeFont:
    font_path = FONTS_DIR / LANGUAGE_FONT_MAP[language]
    if not font_path.exists():
        import subprocess
        subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "download_fonts.py"), "--force"], check=True)
    if not font_path.exists():
        return ImageFont.load_default()
    return ImageFont.truetype(str(font_path), size=size)


def wrap_text(text: str, font: ImageFont.FreeTypeFont, max_width: int) -> list[str]:
    words = text.split()
    if not words: return []
    tmp = ImageDraw.Draw(Image.new("RGB", (1, 1)))
    lines = []
    current = ""
    for word in words:
        test = (current + " " + word).strip() if current else word
        if tmp.textbbox((0, 0), test, font=font)[2] <= max_width:
            current = test
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines


def fit_text(text: str, bbox: list[int], language: str, min_font_size: int = 6) -> tuple[ImageFont.FreeTypeFont, list[str], int]:
    """Fit text into bbox by shrinking font size. Truncates lines if still too tall."""
    x0, y0, x1, y1 = bbox
    box_w, box_h = x1 - x0, y1 - y0
    if box_w <= 0 or box_h <= 0:
        return get_font(language, min_font_size), [text], min_font_size
    
    initial_size = max(min_font_size, int(box_h * 0.75))
    tmp = ImageDraw.Draw(Image.new("RGB", (1, 1)))
    
    for size in range(initial_size, min_font_size - 1, -1):
        font = get_font(language, size)
        lines = wrap_text(text, font, box_w)
        if not lines: continue
        line_h = tmp.textbbox((0, 0), lines[0], font=font)[3]
        spacing = max(1, int(size * 0.15))
        total_h = line_h * len(lines) + spacing * (len(lines) - 1)
        if total_h <= box_h:
            return font, lines, size
    
    # At minimum size, truncate lines that overflow vertically
    font = get_font(language, min_font_size)
    lines = wrap_text(text, font, box_w)
    line_h = tmp.textbbox((0, 0), (lines[0] if lines else "A"), font=font)[3]
    spacing = max(1, int(min_font_size * 0.15))
    max_lines = max(1, box_h // (line_h + spacing))
    return font, lines[:max_lines], min_font_size


def overlay_text(image: Image.Image, original: Image.Image,
                 translations: list[dict], language: str) -> Image.Image:
    """Render translated text onto inpainted image with bbox clipping."""
    result = image.copy()
    tmp = ImageDraw.Draw(Image.new("RGB", (1, 1)))
    
    for trans in translations:
        text = trans["translated"]
        bbox = [int(c) for c in trans["bbox"]]
        x0, y0, x1, y1 = bbox
        box_w, box_h = x1 - x0, y1 - y0
        if not text.strip() or box_w <= 0 or box_h <= 0:
            continue
        
        bg_color = sample_background_color(original, bbox)
        text_color = estimate_text_color(original, bbox, bg_color)
        font, lines, font_size = fit_text(text, bbox, language)
        if not lines: continue
        
        # Draw onto a clipped sub-image to prevent overflow
        text_img = Image.new("RGBA", (box_w, box_h), (0, 0, 0, 0))
        text_draw = ImageDraw.Draw(text_img)
        
        line_h = tmp.textbbox((0, 0), lines[0], font=font)[3]
        spacing = max(1, int(font_size * 0.15))
        total_h = line_h * len(lines) + spacing * (len(lines) - 1)
        y_off = max(0, (box_h - total_h) // 2)
        
        for line in lines:
            if y_off + line_h > box_h:
                break
            text_draw.text((0, y_off), line, fill=text_color, font=font)
            y_off += line_h + spacing
        
        result.paste(text_img, (x0, y0), mask=text_img)
    
    return result


print("Inpainting and overlay stages ready.")

## Run Full Pipeline

Execute the complete pipeline: Input → OCR → Translate → Inpaint → Overlay → Output

In [ ]:
def translate_document(
    input_path: Path,
    target_languages: list[str],
    ocr_engine: str = "easyocr",
    translator_engine: str = "claude",
    inpainting_method: str = "opencv_telea",
    dpi: int = 300,
    page_range: list[int] | None = None,
    output_dir: Path = OUTPUT_DIR / "pipeline",
) -> dict[str, list[Image.Image]]:
    """
    Full end-to-end document translation pipeline.
    
    Args:
        input_path: Path to PDF or image file
        target_languages: List of target languages
        ocr_engine: OCR engine to use
        translator_engine: Translation engine to use
        inpainting_method: Inpainting method to use
        dpi: DPI for PDF rendering
        page_range: Optional list of page indices to process
        output_dir: Directory to save outputs
    
    Returns:
        Dict mapping language → list of translated page images
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    total_start = time.time()
    
    # ── Step 0: Load input ────────────────────────────────────────────────────
    print("=" * 60)
    print("STAGE 0: Loading input...")
    t0 = time.time()
    page_images = load_input(input_path, dpi=dpi)
    
    if page_range:
        page_images = [page_images[i] for i in page_range]
    
    print(f"  Loaded {len(page_images)} page(s) in {time.time()-t0:.1f}s")
    
    # ── Step 1: OCR (run once, reuse for all languages) ───────────────────────
    print("=" * 60)
    print(f"STAGE 1: Running OCR ({ocr_engine})...")
    t0 = time.time()
    all_ocr_results = []
    for i, img in enumerate(tqdm(page_images, desc="OCR")):
        blocks = run_ocr(img, engine=ocr_engine)
        all_ocr_results.append(blocks)
        print(f"  Page {i}: {len(blocks)} text blocks")
    print(f"  OCR complete in {time.time()-t0:.1f}s")
    
    # ── Step 2+3+4: For each language: translate, inpaint, overlay ────────────
    results = {}
    
    for lang in target_languages:
        print("=" * 60)
        print(f"PROCESSING LANGUAGE: {LANGUAGE_CODES[lang]['name'].upper()}")
        
        lang_output_images = []
        
        for page_idx, (original_img, ocr_blocks) in enumerate(zip(page_images, all_ocr_results)):
            print(f"\n  Page {page_idx}:")
            
            # Stage 2: Translate
            print(f"    Translating ({translator_engine})...")
            t0 = time.time()
            translations = translate_blocks(ocr_blocks, lang, engine=translator_engine)
            print(f"    Translated {len(translations)} blocks in {time.time()-t0:.1f}s")
            
            # Stage 3: Inpaint
            print(f"    Inpainting ({inpainting_method})...")
            t0 = time.time()
            inpainted = inpaint_image(original_img, ocr_blocks, method=inpainting_method)
            print(f"    Inpainted in {time.time()-t0:.1f}s")
            
            # Stage 4: Overlay
            print(f"    Overlaying translated text...")
            t0 = time.time()
            final_image = overlay_text(inpainted, original_img, translations, lang)
            print(f"    Overlay complete in {time.time()-t0:.1f}s")
            
            lang_output_images.append(final_image)
            
            # Save individual page
            save_image(final_image, output_dir / f"page_{page_idx}_{lang}.png")
        
        # Save as PDF if multiple pages
        if len(lang_output_images) > 1:
            pdf_path = output_dir / f"translated_{lang}.pdf"
            lang_output_images[0].save(
                str(pdf_path), "PDF", save_all=True,
                append_images=lang_output_images[1:], resolution=dpi,
            )
            print(f"\n  Saved PDF: {pdf_path}")
        elif len(lang_output_images) == 1:
            pdf_path = output_dir / f"translated_{lang}.pdf"
            lang_output_images[0].save(str(pdf_path), "PDF", resolution=dpi)
            print(f"\n  Saved PDF: {pdf_path}")
        
        results[lang] = lang_output_images
    
    total_time = time.time() - total_start
    print("\n" + "=" * 60)
    print(f"PIPELINE COMPLETE in {total_time:.1f}s")
    print(f"Output saved to: {output_dir}")
    
    return results


print("Pipeline function ready.")

## Execute Pipeline

In [ ]:
# Run the full pipeline
results = translate_document(
    input_path=INPUT_FILE,
    target_languages=TARGET_LANGUAGES,
    ocr_engine=OCR_ENGINE,
    translator_engine=TRANSLATOR_ENGINE,
    inpainting_method=INPAINTING_METHOD,
    dpi=DPI,
    page_range=PAGE_RANGE,
    output_dir=PIPELINE_OUTPUT_DIR,
)

## Visual Comparison: Original vs Translated

Side-by-side comparison of the first page in each target language.

In [ ]:
# Load the original first page for comparison
original_pages = load_input(INPUT_FILE, dpi=DPI)
original_first = original_pages[0] if original_pages else None

# Display original vs each language translation
for lang in TARGET_LANGUAGES:
    if lang in results and results[lang]:
        lang_name = LANGUAGE_CODES[lang]["name"]
        display_comparison(
            original_first, results[lang][0],
            "Original (English)", f"Translated ({lang_name})",
            figsize=(18, 10),
        )

In [ ]:
# List all output files
print("Output files:")
for f in sorted(PIPELINE_OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.0f} KB)")

print(f"\n✓ Pipeline complete! Check {PIPELINE_OUTPUT_DIR} for output files.")